# Protect your GitHub Repositories at scale with Terraform

Interactive companion to the talk of the same name.

Terraform does the setup here. The demo itself — opening a pull request,
changing a property — happens in the **GitHub UI**, in a tab next to this one,
with one command here to show the effect.

## What you need

- A GitHub **organization on Team or Enterprise Cloud**. Organization rulesets
  can't be created on a free plan; step 3 checks before anything is created.
- A **classic PAT** with `repo`, `admin:org`, `workflow`, and `delete_repo` for
  the cleanup in step 8.

## Put the token in Colab Secrets

🔑 in the left sidebar → add a secret named `GITHUB_TOKEN` → toggle **Notebook
access** on. A token pasted into a cell gets saved into the notebook file and
travels with it when you share it.

**This creates real resources.** Point it at a throwaway org.

---
## 1. Install Terraform and gh

In [ ]:
%%bash
set -euo pipefail

TERRAFORM_VERSION=1.16.5
GH_VERSION=2.102.0

wget -qO /tmp/terraform.zip \
  "https://releases.hashicorp.com/terraform/${TERRAFORM_VERSION}/terraform_${TERRAFORM_VERSION}_linux_amd64.zip"
unzip -oq /tmp/terraform.zip -d /usr/local/bin

wget -qO /tmp/gh.tar.gz \
  "https://github.com/cli/cli/releases/download/v${GH_VERSION}/gh_${GH_VERSION}_linux_amd64.tar.gz"
tar -xzf /tmp/gh.tar.gz -C /tmp
install -m 755 "/tmp/gh_${GH_VERSION}_linux_amd64/bin/gh" /usr/local/bin/gh

terraform version
gh version

---
## 2. Configuration

Terraform reads `GITHUB_TOKEN`, and `gh` falls back to it, so one variable
covers both.

In [ ]:
import os

try:
    from google.colab import userdata
    os.environ["GITHUB_TOKEN"] = userdata.get("GITHUB_TOKEN")
    print("Token loaded from Colab Secrets.")
except ImportError:
    assert os.environ.get("GITHUB_TOKEN"), "Set GITHUB_TOKEN before running."
    print("Token loaded from the environment.")

**Edit the top of this cell.** It writes the exports to a file that every later
cell sources — each `%%bash` cell is its own shell, so an `export` in one is gone
by the next.

In [ ]:
%%bash
set -euo pipefail

cat > /content/env.sh <<'EOF'
export GITHUB_ORG="my-org"          # <-- edit this
export REPO_NAME="payments-service"
export TEAM_SLUG="security"                     # gets ruleset bypass
export OWNER_TEAM_SLUG="platform-engineering"   # owns the code, in CODEOWNERS

export REPO_URL="https://github.com/missaelcorm/protect-github-repos-at-scale.git"
export REPO_BRANCH="main"
export WORKDIR="/content/protect-github-repos-at-scale"

# Terraform input variables:
export TF_VAR_github_organization="$GITHUB_ORG"
export TF_VAR_repository_name="$REPO_NAME"
export TF_VAR_bypass_team_slug="$TEAM_SLUG"
export TF_VAR_repository_visibility="public"
export TF_VAR_seed_ci_workflow="true"
export TF_VAR_codeowners="[\"@${GITHUB_ORG}/${OWNER_TEAM_SLUG}\"]"

export DEMO_REPO="${GITHUB_ORG}/${REPO_NAME}"
export TF_IN_AUTOMATION=1
EOF

source /content/env.sh
[[ "$GITHUB_ORG" == "my-org" ]] && { echo "!! Set GITHUB_ORG above first."; exit 1; }

echo "org:       $GITHUB_ORG"
echo "demo repo: $DEMO_REPO"

---
## 3. Preflight

In [ ]:
%%bash
set -euo pipefail
source /content/env.sh

echo "user:   $(gh api user --jq '.login')"

# Classic PATs report scopes in a header; fine-grained tokens don't.
SCOPES="$(gh api --include user 2>/dev/null | grep -i '^x-oauth-scopes:' | cut -d' ' -f2- | tr -d '\r' || true)"
echo "scopes: ${SCOPES:-(not reported — fine-grained token?)}"

for s in repo admin:org workflow delete_repo; do
  [[ -n "$SCOPES" && "$SCOPES" != *"$s"* ]] && echo "   missing: $s"
done

PLAN="$(gh api "orgs/${GITHUB_ORG}" --jq '.plan.name // "unknown"')"
echo
echo "org:    $GITHUB_ORG (plan: $PLAN)"

if [[ "$PLAN" == "free" ]]; then
  cat <<'EOF'

!! Free plan. Module 03 will fail:

     403 Upgrade to GitHub Team to enable this feature.

   Organization rulesets need GitHub Team. Modules 00, 01 and 02 still work,
   and so does the custom-property schema — you can show classification, just
   not classification driving enforcement.

   Also on free: rulesets only apply to PUBLIC repos.
EOF
else
  echo "Plan '$PLAN' supports organization rulesets."
fi

---
## 4. Clone the Terraform

In [ ]:
%%bash
set -euo pipefail
source /content/env.sh

rm -rf "$WORKDIR"
git clone --depth 1 --branch "$REPO_BRANCH" "$REPO_URL" "$WORKDIR"
cd "$WORKDIR"
chmod +x scripts/*.sh
ls -1

---
## 5. Apply the modules, in order

`00` the demo repo, `01` the teams, `02` one repo-level ruleset, `03` the
classification schema and org rulesets.

`01` has to come before `02` and `03`: they resolve their bypass team by slug,
and a missing team fails at plan time with `Not Found`.

In [ ]:
%%bash
set -euo pipefail
source /content/env.sh
cd "$WORKDIR/00-setup"

terraform init -input=false -no-color
terraform apply -auto-approve -input=false -no-color

In [ ]:
%%bash
set -euo pipefail
source /content/env.sh

gh repo view "$DEMO_REPO" --json nameWithOwner,visibility,defaultBranchRef,url \
  --jq '"repo:           \(.nameWithOwner)
visibility:     \(.visibility)
default branch: \(.defaultBranchRef.name)
url:            \(.url)"'

`01` wants a `terraform.tfvars` because `teams` is a map of objects. Two teams:
**security** gets ruleset bypass, **platform-engineering** owns the code and is
in CODEOWNERS, which needs write access to count.

In [ ]:
%%bash
set -euo pipefail
source /content/env.sh
cd "$WORKDIR/01-teams-and-permissions"

cat > terraform.tfvars <<EOF
github_organization = "${GITHUB_ORG}"

teams = {
  "${TEAM_SLUG}" = {
    description = "Application and infrastructure security"
    privacy     = "closed"
  }
  "${OWNER_TEAM_SLUG}" = {
    description = "Owns the payments service"
    privacy     = "closed"
  }
}

team_repository_access = {
  owner_on_demo = {
    team_slug  = "${OWNER_TEAM_SLUG}"
    repository = "${REPO_NAME}"
    permission = "push"
  }
}
EOF

terraform init -input=false -no-color
terraform apply -auto-approve -input=false -no-color

echo
echo "CODEOWNERS errors (empty = valid):"
gh api "repos/${DEMO_REPO}/codeowners/errors" --jq '.errors[]?.message' || true

`02` is one ruleset on one repo — the thing that works but needs repeating for
every repo you own. It's a modest baseline on purpose, so the tiers in `03`
visibly add to it.

In [ ]:
%%bash
set -euo pipefail
source /content/env.sh
cd "$WORKDIR/02-repository-ruleset"

terraform init -input=false -no-color
terraform apply -auto-approve -input=false -no-color

`03` is the module the talk is about: a classification schema, plus org rulesets
that match on property *values* rather than repository names. Nothing in it names
the demo repo.

In [ ]:
%%bash
set -euo pipefail
source /content/env.sh
cd "$WORKDIR/03-classification-and-org-rulesets"

terraform init -input=false -no-color
terraform apply -auto-approve -input=false -no-color

---
## 6. Before: a `standard` repo

`tier` defaults to `standard`, so the repo is already classified and already gets
`baseline-protection`.

In [ ]:
%%bash
set -euo pipefail
source /content/env.sh
cd "$WORKDIR/scripts"

./show-rules.sh "$DEMO_REPO"

BRANCH="$(gh api "repos/$DEMO_REPO" --jq '.default_branch')"
gh api "repos/$DEMO_REPO/rules/branches/$BRANCH" --jq '.[].type' \
  | sort -u > /content/types-before.txt

---
## 7. The demo — two things in the UI

Open the repo in a new tab (its URL is in the step 5 output).

### a. Open a pull request

`README.md` → pencil icon → add a line → **Commit changes…** → **Create a new
branch and start a pull request** → **Propose changes**.

The merge box asks for **one approval**.

### b. Reclassify the repository

Repo **Settings → Custom properties** → set `tier` to `regulated` → **Save**.

You're changing the repository, not the pull request.

### c. Come back and run this

In [ ]:
%%bash
set -euo pipefail
source /content/env.sh
cd "$WORKDIR/scripts"

./show-rules.sh "$DEMO_REPO"

BRANCH="$(gh api "repos/$DEMO_REPO" --jq '.default_branch')"
gh api "repos/$DEMO_REPO/rules/branches/$BRANCH" --jq '.[].type' \
  | sort -u > /content/types-after.txt

echo
echo "== new rule types since step 6"
comm -13 /content/types-before.txt /content/types-after.txt | sed 's/^/   + /'

Two kinds of change: new rule types above, and rules that were already there but
got stricter — compare the **strictest values** block with step 6's, where
approvals went from one to two.

Now reload the pull request. Same PR, nobody pushed to it, no `terraform apply` —
and it now asks for two approvals, a code owner, and three checks.

### What just happened

The repository carries `tier = regulated`, and an org ruleset that was already
deployed — and has never heard of this repository by name — started matching it.
Onboarding the next regulated repo is the same one-line property change.

Classification is a property of the repository; enforcement is a function of the
classification.

---
## 8. Cleanup

Set `CONFIRM_ORG` to your org name to arm this cell.

The custom-property **schema** is org-wide: destroying `03` removes `tier`,
`compliance_framework` and `owning_team` for every repo in the org.

In [ ]:
%%bash
set -euo pipefail
source /content/env.sh

CONFIRM_ORG=""   # <-- type your org name here

if [[ "$CONFIRM_ORG" != "$GITHUB_ORG" ]]; then
  echo "Set CONFIRM_ORG=\"$GITHUB_ORG\" in this cell to proceed. Nothing done."
  exit 0
fi

for m in 03-classification-and-org-rulesets 02-repository-ruleset \
         01-teams-and-permissions 00-setup; do
  [[ -d "$WORKDIR/$m/.terraform" ]] || { echo "-- $m: not applied, skipping"; continue; }
  echo "-- destroying $m"
  terraform -chdir="$WORKDIR/$m" destroy -auto-approve -input=false -no-color \
    || echo "   failed — destroying the repo needs the 'delete_repo' scope"
done

---

## Where to go next

In [the repository](https://github.com/missaelcorm/protect-github-repos-at-scale):

- `docs/plan-requirements.md` — what each GitHub plan allows.
- `docs/operational-realities.md` — drift detection, state layout, auth, rate
  limits, plan limits, and using custom properties as a repo inventory.
- `scripts/README.md` — the same demo in a terminal.